In [1]:
import lance
import  pyarrow as pa

from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm.auto import tqdm


/Users/haochengliu/Documents/projects/lance/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import pyarrow.parquet as pq

data_dir = '/Users/haochengliu/Documents/projects/lance/rust/examples/src/wikitext-103-raw-v1/data'


for file in os.listdir(data_dir):
    if file.endswith('.parquet'):
        file_path = os.path.join(data_dir, file)
        parquet_file = pq.ParquetFile(file_path)
        num_rows = parquet_file.metadata.num_rows
        print(f"{file}: {num_rows:,} rows")


validation-00000-of-00001-4c013962448951dd.parquet: 60 rows
test-00000-of-00001-b7859cf6365689a3.parquet: 62 rows
train-00001-of-00002-0bf6d0c487c2e75b.parquet: 14,783 rows
train-00000-of-00002-b755d19de94348c6.parquet: 14,784 rows


In [3]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')

dataset = load_dataset('wikitext', 'wikitext-103-raw-v1')['train']
print(f"Number of rows in dataset: {len(dataset)}")

dataset = load_dataset('wikitext', 'wikitext-103-raw-v1', streaming=True)['train']
dataset = dataset.shuffle(seed=1337)



Number of rows in dataset: 1801350


In [4]:
def tokenize(sample, field='text'):
    return tokenizer(sample[field])['input_ids']
    

In [5]:
def process_samples(dataset, num_samples = 100_000, field='text'):
    cur_sample = 0
    for sample in tqdm(dataset, total=num_samples):
        if cur_sample == num_samples:
            break
        if not sample[field]:
            continue
        # Tokenize the current sample
        tokenized_sample = tokenize(sample, field)
        # Increment the counter
        cur_sample += 1
        if cur_sample % 5000 == 0:
            print(f"saw {cur_sample} rows")

        yield pa.RecordBatch.from_arrays(
            [tokenized_sample],
            names=["input_ids"]
        )

In [6]:
schema = pa.schema([pa.field('input_ids', pa.int64())])


In [7]:
reader = pa.RecordBatchReader.from_batches(
    schema,
    process_samples(dataset, num_samples=500_000, field='text') # For 500K samples
)


In [8]:
# Write the dataset to disk
lance.write_dataset(
    reader,
    "wikitext_500K.lance",
    schema
)


  0%|          | 1/500000 [00:04<576:04:18,  4.15s/it]

OSError: Dataset already exists: wikitext_500K.lance, /Users/haochengliu/Documents/projects/lance/rust/lance/src/dataset/write/insert.rs:282:31

In [63]:
import lance
ds = lance.dataset("wikitext_500k.lance")
#ds = lance.dataset("rust_wikitext_lance_dataset.lance")
table = ds.to_table()
print(f"Number of rows: {len(table)}")
table.to_pandas()


Number of rows: 50449608


,input_ids
0,1675
1,1035
2,5039
3,262
4,13626
...,...
50449603,25279
50449604,796
50449605,796
50449606,220


In [9]:
# read my rust generated data
import lance
ds = lance.dataset("/Users/haochengliu/Documents/projects/lance/rust/examples/rust_wikitext_lance_dataset.lance")
ds = lance.dataset("/Users/haochengliu/Documents/projects/lance/rust/examples/src/rust_wikitext_lance_dataset.lance")
table = ds.to_table()
print(f"Number of rows: {len(table)}")
table.to_pandas()

Number of rows: 100000


,input_ids
0,[]
1,"[347, 9697, 21809, 10307, 366, 383, 1881, 2080..."
2,[]
3,"[35089, 3713, 1526, 3846, 78, 406, 12427, 357,..."
4,[]
...,...
99995,[]
99996,"[383, 1762, 3670, 329, 262, 4471, 373, 366, 96..."
99997,"[7979, 3262, 900, 14936, 329, 38742, 559, 837,..."
99998,[]
